<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Bevezet%C3%A9s%20a%20Nagy%20Nyelvi%20Modellekbe%20(LLM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bevezetés a Nagy Nyelvi Modellekbe (LLM-k)

**Licenc: CC BY-NC-SA 4.0**

A **nagy nyelvi modellek** (Large Language Models, LLMs) - mint a GPT-4, Claude, LLaMA - méretük és képességeik miatt új paradigmát hoztak az AI-ban.

## Mi tesz egy modellt "nagynak"?

| Model | Paraméterek | Training tokens | Év |
|-------|-------------|-----------------|-----|
| GPT-2 | 1.5B | 40B | 2019 |
| GPT-3 | 175B | 300B | 2020 |
| LLaMA 2 | 7-70B | 2T | 2023 |
| GPT-4 | ~1.8T (becsült) | ~13T | 2023 |

## LLM képességek

Az LLM-ek meglepő képességeket mutatnak:
- **Szöveg generálás**: Koherens, hosszú szövegek
- **Kérdés-válasz**: Tudásbázis nélkül is
- **Fordítás**: Nyelvek között
- **Kód írás**: Programozás segítése
- **Reasoning**: Logikai következtetés
- **In-context learning**: Tanulás példákból

## Tartalomjegyzék
1. Scaling Laws
2. Emergent Abilities
3. RLHF: Human Alignment
4. LLM használat (API)
5. Sampling stratégiák

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Scaling Laws

A **Chinchilla scaling laws** (Hoffmann et al., 2022) megmutatta: a teljesítmény **előre jelezhető** a modell méret, adat mennyiség és compute alapján.

### Power Law

A validation loss közelítőleg:

$$L(N, D) = \left(\frac{N_c}{N}\right)^{\alpha_N} + \left(\frac{D_c}{D}\right)^{\alpha_D} + L_{\infty}$$

Ahol:
- $N$: paraméterek száma
- $D$: training tokenek száma
- $\alpha_N, \alpha_D \approx 0.34$

### Compute-Optimal Training

**Chinchilla megállapítás**: Az optimális paraméter/adat arány körülbelül 1:20.
- 10B paraméter → ~200B token
- GPT-3 (175B param, 300B token) **undertrained** volt!

### Implikációk

1. **Nagyobb adat > nagyobb modell** (ha költségvetés fix)
2. **Előre jelezhető** teljesítmény → tervezhetőség
3. **Nincs szaturáció**: A görbék nem laposodnak el

In [ ]:
# Chinchilla scaling laws vizualizáció
params = np.logspace(8, 12, 20)  # 100M to 1T
loss = 2.5 * (params ** -0.076)  # Approximált scaling

plt.figure(figsize=(8, 4))
plt.loglog(params, loss, 'b-', linewidth=2)
plt.xlabel('Paraméterek')
plt.ylabel('Validation Loss')
plt.title('Scaling Law: Több paraméter → Alacsonyabb loss')

# Modellek jelölése
models = {'GPT-2': 1.5e9, 'GPT-3': 175e9, 'GPT-4': 1e12}
for name, p in models.items():
    plt.axvline(p, linestyle='--', alpha=0.5)
    plt.text(p, 0.8, name, rotation=90)

plt.grid(True, alpha=0.3)
plt.show()

## 2. Emergent Abilities

Az **emergent abilities** olyan képességek, amelyek kis modelleknél nincsenek jelen, de egy kritikus méret felett hirtelen "megjelennek".

### Példák emergent abilities-re

| Képesség | Leírás | Megjelenés (~params) |
|----------|--------|---------------------|
| **In-context learning** | Tanulás példákból prompt-ban | ~10B |
| **Chain-of-thought** | Lépésről lépésre gondolkodás | ~100B |
| **Code generation** | Programkód írás | ~10B |
| **Multi-step reasoning** | Összetett logika | ~100B |

### Miért történik ez?

Vitatott kérdés:
1. **Phase transition**: Kritikus méret után ugrásszerű javulás
2. **Smooth scaling**: Valójában csak a metrika diszkrét (pl. exact match)
3. **Emergent complexity**: Több egyszerű képesség kombinációja

### In-context learning

Az LLM a **prompt-ban lévő példákból tanul**, súlyok módosítása nélkül:

In [ ]:
# In-context learning példa
prompt = """
Translate English to French:

cat → chat
dog → chien
house → maison
car → ???
"""

print("In-context learning (few-shot):")
print(prompt)
print("LLM válasz: voiture")

In [ ]:
# Chain-of-thought prompting
cot_prompt = """
Q: Ha 3 alma van és elveszek 2-t, majd kapok 5-öt, hány almám van?

A: Gondoljuk végig lépésről lépésre:
1. Kezdetben: 3 alma
2. Elveszek 2-t: 3 - 2 = 1 alma
3. Kapok 5-öt: 1 + 5 = 6 alma

Válasz: 6 alma
"""
print("Chain-of-Thought prompting:")
print(cot_prompt)

## 3. RLHF: Reinforcement Learning from Human Feedback

A **RLHF** (Ouyang et al., 2022) az LLM-ek "igazítása" emberi preferenciákhoz.

### A probléma

A pre-trained LLM:
- Jól generál szöveget
- DE: nem követi az utasításokat
- DE: lehet toxikus, helytelen, veszélyes

### RLHF pipeline

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  Pre-training   │───→│      SFT        │───→│      RLHF       │
│  (Huge corpus)  │    │  (Instructions) │    │ (Human prefs)   │
└─────────────────┘    └─────────────────┘    └─────────────────┘
     Base model        Instruction-following   Human-aligned
```

1. **Pre-training**: Nyelvi modell tanítás hatalmas korpuszon
2. **Supervised Fine-Tuning (SFT)**: Instruction-response párokra fine-tuning
3. **Reward Modeling**: Emberi preferenciákból reward modell tanítás
4. **PPO Training**: Reinforcement learning a reward model alapján

### Reward Model

Az emberi annotátorok **páronként rangsorolják** a válaszokat:

```
Prompt: "Mi a főváros?"
Response A: "Budapest a főváros."  ← Jobb
Response B: "Nem tudom."
```

A reward model megtanulja: $r(A) > r(B)$

In [ ]:
# RLHF pipeline vizualizáció
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')

stages = [
    ('Pre-training\n(Web corpus)', 'lightblue'),
    ('SFT\n(Instructions)', 'lightgreen'),
    ('Reward Model\n(Human prefs)', 'lightyellow'),
    ('PPO\n(RL training)', 'lightcoral'),
    ('ChatGPT\n(Final)', 'plum')
]

for i, (text, color) in enumerate(stages):
    x = 0.1 + i * 0.18
    ax.add_patch(plt.Rectangle((x, 0.3), 0.14, 0.4, facecolor=color, edgecolor='black'))
    ax.text(x + 0.07, 0.5, text, ha='center', va='center', fontsize=9)
    if i < len(stages) - 1:
        ax.annotate('', xy=(x + 0.16, 0.5), xytext=(x + 0.14, 0.5),
                   arrowprops=dict(arrowstyle='->'))

plt.title('RLHF Pipeline', fontsize=12, fontweight='bold', pad=20)
plt.show()

## 4. LLM használat (API)

A legtöbb LLM-et **API-n keresztül** érjük el (OpenAI, Anthropic, Google, stb.)

### Chat Completion formátum

A modern LLM-ek **chat formátumot** használnak:

```json
{
  "messages": [
    {"role": "system", "content": "Te egy segítőkész asszisztens vagy."},
    {"role": "user", "content": "Mi a gépi tanulás?"},
    {"role": "assistant", "content": "A gépi tanulás..."}
  ]
}
```

### Role-ok

| Role | Funkció |
|------|---------|
| `system` | Viselkedés beállítása, persona |
| `user` | Felhasználói üzenet |
| `assistant` | Modell válasz (előző válaszok kontextusként) |

In [ ]:
print("""
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Te egy segítőkész asszisztens vagy."},
        {"role": "user", "content": "Mi a gépi tanulás?"}
    ],
    temperature=0.7,
    max_tokens=200
)

print(response.choices[0].message.content)
""")

## 5. Sampling stratégiák és hiperparaméterek

A generálás során a következő token kiválasztása **sampling**-gel történik.

### Kulcs paraméterek

| Paraméter | Hatás | Értékek |
|-----------|-------|---------|
| **temperature** | Randomness | 0=determinisztikus, 1+=kreatív |
| **top_p** (nucleus) | Valószínűség küszöb | 0.9 = top 90% prob mass |
| **top_k** | Top-k token | 50 = csak top 50 token |
| **max_tokens** | Maximum hossz | Response limit |
| **presence_penalty** | Új témák | -2.0 to 2.0 |
| **frequency_penalty** | Ismétlés büntetése | -2.0 to 2.0 |

### Temperature intuíció

- **T=0**: Mindig a legvalószínűbb token → determinisztikus, unalmas
- **T=0.7**: Enyhe variáció → kreatív, de koherens
- **T=1.0**: Eredeti eloszlás → változatos
- **T=2.0**: Nagyon flat eloszlás → random, inkoherens

In [ ]:
# Temperature hatása
def sample_with_temperature(logits, temperature):
    if temperature == 0:
        return torch.argmax(logits)
    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, 1)

logits = torch.tensor([2.0, 1.0, 0.5, 0.1, 0.1])
temps = [0.1, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 4, figsize=(12, 2.5))
for ax, t in zip(axes, temps):
    probs = torch.softmax(logits / t, dim=-1).numpy()
    ax.bar(range(5), probs)
    ax.set_title(f'T={t}')
    ax.set_ylim(0, 1)
plt.suptitle('Temperature hatása a token eloszlásra')
plt.tight_layout()
plt.show()

## Összefoglalás

### Főbb tanulságok

1. **Scaling laws**: A teljesítmény előre jelezhető méret alapján
2. **Emergent abilities**: Új képességek nagy méretnél
3. **RLHF**: Human alignment reinforcement learning-gel
4. **Prompting**: In-context learning, chain-of-thought
5. **Sampling**: Temperature, top_p kontrollálja a kreativitást

### LLM modellek összehasonlítása

| Modell | Fejlesztő | Hozzáférés | Erősség |
|--------|-----------|------------|---------|
| GPT-4 | OpenAI | API | Általános, reasoning |
| Claude | Anthropic | API | Hosszú kontextus, safety |
| LLaMA 2 | Meta | Nyílt súlyok | Lokális futtatás |
| Gemini | Google | API | Multimodális |
| Mixtral | Mistral | Nyílt | MoE architektúra |

### Korlátozások

- **Hallucination**: Kitalált "tények" generálása
- **Knowledge cutoff**: Csak training adatig tud
- **Context length**: Korlátozott ablak méret
- **Reasoning**: Komplex logikában gyenge lehet

### Következő lépések

A következő notebookban a **Chatbot** építést nézzük, ahol az LLM-eket interaktív alkalmazásokban használjuk!